# Adaptive Maintenance AI — Data Exploration

This notebook explores the NASA CMAPSS turbofan degradation dataset and validates
the preprocessing pipeline before training the LSTM model.

**Dataset:** NASA CMAPSS FD001–FD004  
**Reference:** Saxena, A. et al. (2008) — *Damage Propagation Modeling for Aircraft Engine Run-to-Failure Simulation*

In [ ]:
import sys
sys.path.insert(0, '../backend')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Imports OK')

## 1. Load Dataset

In [ ]:
from data.data_loader import DataLoader

loader = DataLoader(data_dir='../data/raw', seq_len=30, max_rul=125)
X_train, y_train, X_test, y_test = loader.load('FD001')

print(f'X_train shape : {X_train.shape}')   # (N, 30, 14)
print(f'y_train shape : {y_train.shape}')   # (N,)
print(f'X_test  shape : {X_test.shape}')
print(f'y_test  shape : {y_test.shape}')

## 2. RUL Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_train, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('RUL Distribution — Train')
axes[0].set_xlabel('RUL (cycles)')

axes[1].hist(y_test, bins=30, color='coral', edgecolor='white')
axes[1].set_title('RUL Distribution — Test')
axes[1].set_xlabel('RUL (cycles)')
plt.tight_layout()
plt.show()

## 3. Sensor Correlation Heatmap

In [ ]:
from data.data_loader import FEATURE_COLS

# Use last timestep of each window for correlation
last_ts = X_train[:, -1, :]   # (N, 14)
corr_df = pd.DataFrame(last_ts, columns=FEATURE_COLS)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.3, square=True, annot_kws={'size': 8})
plt.title('Sensor Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 4. LSTM Training (Quick Smoke Test)

In [ ]:
from models.lstm_model import LSTMModel

model = LSTMModel(input_size=len(FEATURE_COLS), seq_len=30, hidden_size=64, num_layers=1)
history = model.fit(X_train[:500], y_train[:500], epochs=5, batch_size=32)

plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'],   label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('LSTM Training Curve (smoke test)')
plt.legend()
plt.show()

## 5. Anomaly Detector Validation

In [ ]:
from models.anomaly_detector import AnomalyDetector

ad = AnomalyDetector(contamination=0.05)
baseline = X_train[:, -1, :]   # use last timestep
ad.fit(baseline)

normal_score = ad.score(baseline[0])
fault_vec    = baseline[0] * 10   # simulate fault spike
fault_score  = ad.score(fault_vec)

print(f'Normal anomaly score : {normal_score:.4f}')
print(f'Fault  anomaly score : {fault_score:.4f}')
assert fault_score > normal_score, 'Anomaly detector should score faults higher'
print('✓ Anomaly detector validation passed')